# 01 — Load & Clean
This notebook loads raw oil price data from the EIA and merges it into a 
single master DataFrame for use in downstream analysis.

**Data sources:**
- Europe Brent Spot Price (daily) — EIA / Thomson Reuters
- Cushing OK WTI Spot Price (daily) — EIA / Thomson Reuters
- US Retail Gasoline Price (weekly) — EIA

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Raw Data
Each EIA file contains 4 rows of metadata before the actual headers. 
We skip these and rename columns to clean, consistent names.

In [2]:
# Load raw data - skip 4 rows of EIA metadata
brent = pd.read_csv('../data/Europe_Brent_Spot_Price_FOB.csv', skiprows=4)
wti = pd.read_csv('../data/Cushing_OK_WTI_Spot_Price_FOB.csv', skiprows=4)
gas = pd.read_csv('../data/Weekly_U.S._All_Grades_All_Formulations_Retail_Gasoline_Prices.csv', skiprows=4)

# Rename columns to something clean 
brent.columns = ['date', 'brent']
wti.columns = ['date', 'wti']
gas.columns = ['date', 'gas']

# Parse dates
brent['date'] = pd.to_datetime(brent['date'])
wti['date'] = pd.to_datetime(wti['date'])
gas['date'] = pd.to_datetime(gas['date'])

# Preview each
print(brent.head())
print(wti.head())
print(gas.head())

        date   brent
0 2026-03-16  101.04
1 2026-03-13  103.23
2 2026-03-12  102.38
3 2026-03-11   90.98
4 2026-03-10   89.84
        date    wti
0 2026-03-16  93.39
1 2026-03-13  98.48
2 2026-03-12  95.61
3 2026-03-11  86.80
4 2026-03-10  83.71
        date    gas
0 2026-03-02  3.148
1 2026-02-23  3.072
2 2026-02-16  3.057
3 2026-02-09  3.033
4 2026-02-02  2.995


## Merge & Clean
We merge all three price series on date using an outer join to preserve 
all available dates. Weekend and holiday gaps are forward-filled since 
prices don't change on non-trading days.

In [3]:
# Merge all three on date using outer join to keep all dates
master = pd.merge(brent, wti, on='date', how='outer')
master = pd.merge(master, gas, on='date', how='outer')

# Sort by date ascending
master = master.sort_values('date').reset_index(drop=True)

# Forward fill weekend/holiday gaps
master = master.ffill()

# Drop any remaining nulls (before data collection started)
master = master.dropna()

# Preview
print(master.shape)
print(master.head())
print(master.tail())

(8513, 4)
           date  brent    wti    gas
1858 1993-04-05  18.95  20.59  1.068
1859 1993-04-06  18.65  20.33  1.068
1860 1993-04-07  18.70  20.35  1.068
1861 1993-04-08  18.53  20.22  1.068
1862 1993-04-12  18.53  20.43  1.079
            date   brent    wti    gas
10366 2026-03-10   89.84  83.71  3.148
10367 2026-03-11   90.98  86.80  3.148
10368 2026-03-12  102.38  95.61  3.148
10369 2026-03-13  103.23  98.48  3.148
10370 2026-03-16  101.04  93.39  3.148


## Save Master Dataset
Export the cleaned, merged DataFrame to CSV for use in downstream notebooks.

In [4]:
master.to_csv('../outputs/master_prices.csv', index=False)
print("Saved")

Saved


## Join Events
We join the events table onto the master price DataFrame so each event 
date is flagged in the timeline.

In [5]:
events = pd.read_csv('../data/events.csv', parse_dates=['date'])

# Merge events onto master on date (left join to keep all price rows)
master = pd.merge(master, events[['date', 'name', 'type']], on='date', how='left')

# Preview rows where events fall
print(master[master['name'].notna()])

           date   brent     wti    gas                   name       type
1508 1999-02-01   10.81   12.36  0.971        Oil price crash      crash
2185 2001-09-11   29.12   27.65  1.562           9/11 attacks  terrorism
2578 2003-03-20   28.00   28.62  1.768          Iraq invasion   invasion
3947 2008-07-03  143.95  145.31  4.146  2008 Financial Crisis  financial
4620 2011-02-17  103.45   85.05  3.193        Libya civil war  civil war
7463 2022-02-24  101.29   92.77  3.624       Ukraine invasion   invasion
8318 2025-06-13   76.00   73.84  3.235    Israel-Iran strikes    strikes
8502 2026-03-02   77.24   71.13  3.148  US-Israel war on Iran        war


## Diagnostic Check — Missing Events
5 of 7 events matched successfully. Two events are missing from the merged data:
- **Gulf War (1990-08-02)** — may predate our Brent price data which starts in 1993
- **US-Israel war on Iran (2026-02-28)** — Feb 28 may be missing due to a trading gap

We check the earliest available date and inspect the dates around Feb 28 2026 
to determine whether to drop or substitute these events.

In [6]:
# Check earliest date in master
print("Earliest date:", master['date'].min())

# Check if Feb 28 2026 exists
print(master[master['date'] == '2026-02-28'])

# Check dates around Feb 28 2026
print(master[master['date'].between('2026-02-01', '2026-03-09')])

Earliest date: 1993-04-05 00:00:00
Empty DataFrame
Columns: [date, brent, wti, gas, name, type]
Index: []
           date  brent    wti    gas                   name type
8482 2026-02-02  67.72  61.60  2.995                    NaN  NaN
8483 2026-02-03  70.01  62.62  2.995                    NaN  NaN
8484 2026-02-04  71.15  64.56  2.995                    NaN  NaN
8485 2026-02-05  69.87  62.90  2.995                    NaN  NaN
8486 2026-02-06  70.45  63.77  2.995                    NaN  NaN
8487 2026-02-09  71.19  64.53  3.033                    NaN  NaN
8488 2026-02-10  71.01  64.20  3.033                    NaN  NaN
8489 2026-02-11  71.52  64.80  3.033                    NaN  NaN
8490 2026-02-12  69.80  63.08  3.033                    NaN  NaN
8491 2026-02-13  69.96  63.05  3.033                    NaN  NaN
8492 2026-02-16  70.81  63.05  3.057                    NaN  NaN
8493 2026-02-17  69.77  62.53  3.057                    NaN  NaN
8494 2026-02-18  71.78  65.33  3.057             

## Fix & Re-merge Events
Feb 28 2026 is a Saturday (no trading data) so we update the Iran event 
to Mar 2 — the first trading day after the strikes began. We also reload 
master clean before re-merging to avoid duplicate columns.

In [7]:
# Reload clean master
master = pd.read_csv('../outputs/master_prices.csv', parse_dates=['date'])

# Reload updated events
events = pd.read_csv('../data/events.csv', parse_dates=['date'])

# Merge
master = pd.merge(master, events[['date', 'name', 'type']], on='date', how='left')

# Confirm all 7 events matched
print(master[master['name'].notna()])

           date   brent     wti    gas                   name       type
1508 1999-02-01   10.81   12.36  0.971        Oil price crash      crash
2185 2001-09-11   29.12   27.65  1.562           9/11 attacks  terrorism
2578 2003-03-20   28.00   28.62  1.768          Iraq invasion   invasion
3947 2008-07-03  143.95  145.31  4.146  2008 Financial Crisis  financial
4620 2011-02-17  103.45   85.05  3.193        Libya civil war  civil war
7463 2022-02-24  101.29   92.77  3.624       Ukraine invasion   invasion
8318 2025-06-13   76.00   73.84  3.235    Israel-Iran strikes    strikes
8502 2026-03-02   77.24   71.13  3.148  US-Israel war on Iran        war


## Save Final Master Dataset
Export the fully merged DataFrame — price data + event labels — to CSV 
for use in all downstream notebooks.

In [8]:
master.to_csv('../outputs/master_prices.csv', index=False)
print("Saved! Shape:", master.shape)

Saved! Shape: (8513, 6)
